## Day 3 

### Tensor Basics

#### .shape

Shape gives us the dimensions of the array

#### .dtype

This gives us the data type of the array. Could be anything like int32, float32, int64.

#### .device

Tells us if the array is on thg cpu or gpu

In [1]:
import torch

x = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

print(x.shape)   
print(x.dtype)   
print(x.device)

torch.Size([2, 3])
torch.float32
cpu


#### tensor.to(device)

this command is used to move things between cpu and gpu.
A tensor is not truly useful to us unless it is performing operations in the GPU.  

After doing our calculations, we need to move the tensor back to the CPU, so we use .cpu() here  



In [2]:
x = torch.tensor([1.0, 2.0, 3.0])  # starts on CPU
print(x.device)  # cpu

x = x.to("cuda")                    # move to GPU
print(x.device)  # cuda:0

y = x.cpu()            #back to cpu

cpu
cuda:0


#### .numpy()

Tensors are pytorch specific.  
After doing our calculations, if we want to do some other things with the data like plot graphs, we cant with tensors. Most of the libraries support numpy. So we convert it to numpy.  
But we need to transfer tensor from gpu to cpu before we convert to numpy, otherwise error.

In [3]:
x = torch.tensor([1.0, 2.0]).to("cuda")  # on GPU
y = x.cpu()                               # back on CPU

x = torch.tensor([1.0, 2.0])
arr = x.numpy()           # now numpy array
print(type(arr))          

<class 'numpy.ndarray'>


### COmparing matmul between 1000, 2500, and 10000 multiplications

In [4]:
import torch
import time

size = 1000

a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

print(f"Matrices created. Shape: {a_cpu.shape}, dtype: {a_cpu.dtype}, device: {a_cpu.device}")

# --- CPU timing ---
start = time.time()
result_cpu = a_cpu @ b_cpu        # @ is the matmul operator
cpu_time = time.time() - start
print(f"\nCPU time: {cpu_time:.4f} seconds")

# --- GPU timing ---
# Move tensors to GPU
a_gpu = a_cpu.to('cuda')
b_gpu = b_cpu.to('cuda')

print(f"\nMoved to GPU. Device: {a_gpu.device}")

# Warm-up run (first GPU op includes setup time — we throw this away)
_ = a_gpu @ b_gpu
torch.cuda.synchronize()

# Real GPU timing
start = time.time()
result_gpu = a_gpu @ b_gpu
torch.cuda.synchronize()
gpu_time = time.time() - start
print(f"GPU time: {gpu_time:.4f} seconds")

# --- The verdict ---
print(f"\n🏁 GPU was {cpu_time / gpu_time:.1f}x faster than CPU")

# Clean up after the cell
del a_cpu, b_cpu, result_cpu, a_gpu, b_gpu, result_gpu, _

import gc
gc.collect()
torch.cuda.empty_cache()

Matrices created. Shape: torch.Size([1000, 1000]), dtype: torch.float32, device: cpu

CPU time: 0.0083 seconds

Moved to GPU. Device: cuda:0
GPU time: 0.0026 seconds

🏁 GPU was 3.2x faster than CPU


### 2500 size

In [5]:
import torch
import time

size = 2500

a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

print(f"Matrices created. Shape: {a_cpu.shape}, dtype: {a_cpu.dtype}, device: {a_cpu.device}")

# --- CPU timing ---
start = time.time()
result_cpu = a_cpu @ b_cpu        # @ is the matmul operator
cpu_time = time.time() - start
print(f"\nCPU time: {cpu_time:.4f} seconds")

# --- GPU timing ---
# Move tensors to GPU
a_gpu = a_cpu.to('cuda')
b_gpu = b_cpu.to('cuda')

print(f"\nMoved to GPU. Device: {a_gpu.device}")

# Warm-up run (first GPU op includes setup time — we throw this away)
_ = a_gpu @ b_gpu
torch.cuda.synchronize()

# Real GPU timing
start = time.time()
result_gpu = a_gpu @ b_gpu
torch.cuda.synchronize()
gpu_time = time.time() - start
print(f"GPU time: {gpu_time:.4f} seconds")

# --- The verdict ---
print(f"\n🏁 GPU was {cpu_time / gpu_time:.1f}x faster than CPU")

# Clean up after the cell
del a_cpu, b_cpu, result_cpu, a_gpu, b_gpu, result_gpu, _

import gc
gc.collect()
torch.cuda.empty_cache()

Matrices created. Shape: torch.Size([2500, 2500]), dtype: torch.float32, device: cpu

CPU time: 0.1629 seconds

Moved to GPU. Device: cuda:0
GPU time: 0.0223 seconds

🏁 GPU was 7.3x faster than CPU


### 10000 Size

In [6]:
import torch
import time

size = 10000

a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

print(f"Matrices created. Shape: {a_cpu.shape}, dtype: {a_cpu.dtype}, device: {a_cpu.device}")

# --- CPU timing ---
start = time.time()
result_cpu = a_cpu @ b_cpu        # @ is the matmul operator
cpu_time = time.time() - start
print(f"\nCPU time: {cpu_time:.4f} seconds")

# --- GPU timing ---
# Move tensors to GPU
a_gpu = a_cpu.to('cuda')
b_gpu = b_cpu.to('cuda')

print(f"\nMoved to GPU. Device: {a_gpu.device}")

# Warm-up run (first GPU op includes setup time — we throw this away)
_ = a_gpu @ b_gpu
torch.cuda.synchronize()

# Real GPU timing
start = time.time()
result_gpu = a_gpu @ b_gpu
torch.cuda.synchronize()
gpu_time = time.time() - start
print(f"GPU time: {gpu_time:.4f} seconds")

# --- The verdict ---
print(f"\n🏁 GPU was {cpu_time / gpu_time:.1f}x faster than CPU")

# Clean up after the cell
del a_cpu, b_cpu, result_cpu, a_gpu, b_gpu, result_gpu, _

import gc
gc.collect()
torch.cuda.empty_cache()

Matrices created. Shape: torch.Size([10000, 10000]), dtype: torch.float32, device: cpu

CPU time: 10.7306 seconds

Moved to GPU. Device: cuda:0
GPU time: 1.2389 seconds

🏁 GPU was 8.7x faster than CPU


### Conclusion

We can see that as the size increases, the speed of GPU increases compared to CPU. For small n, they are roughly equal. But for bigger n, the GPU performs much better.  

I ran these codes a few times and I had gotten unconsistent results.  

Sometimes 1000 would be faster and sometimes 10000.  

I had already taken into consideration the GPU warm up time.

The main reason I suspect:  

We are comparing the time taken between CPU and GPU. We are not comparing the GPU time taken for each operation.  

The 1000 matmul is too small to quantify properly. Maybe the CPU is preocupied with some tasks, so it queues up the multiplication program. The GPU is mostly free so it shows consistent results. The CPU may sometimes take extra time during 1000 matmul because it may be occupied with previous tasks. So the CPU has taken more time. Thats why it shows it was much faster than GPU time.  

For the smaller 1000 array, even a small change in timing can cause a big error.  

Like even a 5ms change in 1000 array will cause a huge error.  

But for 10000 array, which can for approx 12 seconds, the 5ms delay caused by the CPU is negligible.